In [9]:
from google.colab import files

uploaded = files.upload()

Saving cleaned_synthetic_student_dataset (1).csv to cleaned_synthetic_student_dataset (1) (1).csv
Saving p1_synthetic_test.csv to p1_synthetic_test.csv
Saving p1_synthetic_train.csv to p1_synthetic_train.csv
Saving p1_synthetic_val.csv to p1_synthetic_val.csv
Saving synthetic_student_dataset.csv to synthetic_student_dataset.csv


In [10]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [11]:
train = pd.read_csv('/content/p1_synthetic_train.csv')

train = train.dropna(
    subset=[
        'mastery_score',
        'decay_rate',
        'difficulty',
        'response_correct',
        'bloom_tier'
    ]
)

features = [
    'decay_rate',
    'difficulty',
    'response_correct',
    'bloom_tier'
]

X_train = train[features]
y_train = train['mastery_score']

print(train.shape)

(29629, 9)


In [12]:
test = pd.read_csv('/content/p1_synthetic_test.csv')

test = test.dropna(
    subset=[
        'mastery_score',
        'decay_rate',
        'difficulty',
        'response_correct',
        'bloom_tier'
    ]
)

X_test = test[features]
y_test = test['mastery_score']

print(test.shape)

(6383, 9)


In [17]:
!pip install xgboost

In [18]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_test)

acc_xgb = accuracy_score(y_test, pred_xgb)
f1_xgb = f1_score(y_test, pred_xgb)

print("Accuracy:", acc_xgb)
print("F1 Score:", f1_xgb)

Accuracy: 0.8491305028983237
F1 Score: 0.918272086904863


In [19]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline RF",
        "Balanced RF",
        "XGBoost"
    ],
    "Accuracy": [
        0.8267,
        0.8344,
        acc_xgb
    ],
    "F1 Score": [
        0.9044,
        0.9091,
        f1_xgb
    ]
})

print(comparison)

         Model  Accuracy  F1 Score
0  Baseline RF  0.826700  0.904400
1  Balanced RF  0.834400  0.909100
2      XGBoost  0.849131  0.918272


In [20]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, pred_xgb))

print(confusion_matrix(y_test, pred_xgb))

              precision    recall  f1-score   support

         0.0       0.43      0.01      0.02       960
         1.0       0.85      1.00      0.92      5423

    accuracy                           0.85      6383
   macro avg       0.64      0.50      0.47      6383
weighted avg       0.79      0.85      0.78      6383

[[  10  950]
 [  13 5410]]


In [21]:
import joblib

joblib.dump(xgb, "p1_xgboost_best.pkl")

print("Best model saved")

Best model saved
